# Qwen2.5-Coder-7B QLoRA Fine-tuning (Optimized)

This notebook performs QLoRA PEFT on `Qwen/Qwen2.5-Coder-7B` using the `hoangnh39/magicoder_samples_qwen2.5-coder` dataset.

In [1]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

In [2]:
# pip installs
!pip install -q -U bitsandbytes transformers peft accelerate datasets trl sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 103.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 93.9 MB/s eta 0:00:00:00:01


In [3]:
# imports
import os
import re
import math
import torch
import transformers
import wandb
from tqdm import tqdm
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig, set_seed
from datasets import load_dataset
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer, SFTConfig
from datetime import datetime

In [4]:
# set check point tracker
def create_checkpoint_tracker():
    checkpoint_file = "checkpoint_tracker.py"
    with open(checkpoint_file, "w") as f:
        f.write("""
def get_latest_step():
    try:
        with open("latest_step.txt", "r") as f:
            return int(f.read().strip())
    except:
        return 0

def save_latest_step(step):
    with open("latest_step.txt", "w") as f:
        f.write(str(step))
""")

create_checkpoint_tracker()
from checkpoint_tracker import get_latest_step, save_latest_step

In [5]:
def train_or_resume(
    base_model_name="Qwen/Qwen2.5-Coder-7B",
    hf_model_name="hoangnh39/qwen2.5-coder-7b-magicoder",
    train_dataset="hoangnh39/magicoder_samples_qwen2.5-coder",
    lora_config=None,
    steps_per_session=500,
    max_total_steps=1000,
    batch_size=1,
    grad_accum_steps=16,
    save_steps=20
):
    """
    Train a model or resume training from the latest checkpoint on Hugging Face.
    """
    latest_step = get_latest_step()

    if latest_step >= max_total_steps:
        print(f"Training already completed! Reached {latest_step}/{max_total_steps} steps")
        return

    steps_this_session = min(steps_per_session, max_total_steps - latest_step)
    print(f"Training for {steps_this_session} steps (total progress: {latest_step}/{max_total_steps})")

    # Handle 'resume' keyword as per instructions
    is_resuming = (base_model_name == 'resume') or (latest_step > 0)
    actual_base_model = "Qwen/Qwen2.5-Coder-7B" if base_model_name == 'resume' else base_model_name

    # 1. Config 4-bit quantization
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4"
    )

    # 2. Load Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(actual_base_model, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # 3. Load Model
    try:
        if is_resuming:
            print(f"Resuming from checkpoint/adapter: {hf_model_name}")
            model = AutoModelForCausalLM.from_pretrained(
                actual_base_model,
                quantization_config=quant_config,
                device_map="auto",
                trust_remote_code=True,
                torch_dtype=torch.bfloat16
            )
            model = PeftModel.from_pretrained(model, hf_model_name, is_trainable=True)
        else:
            print(f"Starting fresh training from: {actual_base_model}")
            model = AutoModelForCausalLM.from_pretrained(
                actual_base_model,
                quantization_config=quant_config,
                device_map="auto",
                trust_remote_code=True,
                torch_dtype=torch.bfloat16
            )
            model = prepare_model_for_kbit_training(model)
    except Exception as e:
        print(f"Error loading model, attempting to start fresh: {e}")
        model = AutoModelForCausalLM.from_pretrained(
            actual_base_model,
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=True,
            torch_dtype=torch.bfloat16
        )
        model = prepare_model_for_kbit_training(model)

    # 4. LoRA Configuration
    if lora_config is None:
        lora_config = LoraConfig(
            r=16,
            lora_alpha=32,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM",
        )

    # 5. Load Dataset
    print(f"Loading dataset: {train_dataset}...")
    dataset = load_dataset(train_dataset, split="train")

    # Formatting function for Instruction Fine-tuning
    def format_prompts(example):
        output_texts = []
        for i in range(len(example['instruction'])):
            text = f"### Instruction:\n{example['instruction'][i]}\n\n### Response:\n{example['output'][i]}"
            output_texts.append(text)
        return output_texts

    # 6. SFT Configuration
    train_params = SFTConfig(
        output_dir="./checkpoints",
        num_train_epochs=1,
        max_steps=steps_this_session,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum_steps,
        optim="paged_adamw_32bit",
        save_steps=save_steps,
        logging_steps=20,
        learning_rate=1e-4,
        weight_decay=0.001,
        fp16=False,
        bf16=True,
        max_grad_norm=0.3,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        push_to_hub=True,
        hub_model_id=hf_model_name,
        hub_private_repo=True,
        report_to="wandb" if os.environ.get("WANDB_API_KEY") else "none",
        max_length=512,
    )

    # 7. Trainer Setup
    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        peft_config=None if is_resuming else lora_config,
        processing_class=tokenizer,
        args=train_params,
        # formatting_func=format_prompts,
    )

    # 8. Train & Push
    trainer.train()
    trainer.model.push_to_hub(hf_model_name, private=True)
    tokenizer.push_to_hub(hf_model_name, private=True)

    # 9. Update step count
    save_latest_step(latest_step + steps_this_session)
    print(f"Completed session: {latest_step + steps_this_session}/{max_total_steps} steps")

    return latest_step + steps_this_session

In [ ]:
# Login and Config
hf_token = user_secrets.get_secret("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

# Optional: WandB
try:
    wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")
    if wandb_api_key:
        os.environ["WANDB_API_KEY"] = wandb_api_key
        wandb.login()
        os.environ["WANDB_PROJECT"] = "qwen2.5-coder-magicoder"
except:
    print("WandB API key not found, skipping logging.")

# LoRA Parameters
lora_parameters = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Execution
current_progress = train_or_resume(
    base_model_name="Qwen/Qwen2.5-Coder-7B",
    hf_model_name="hoangnh39/qwen2.5-coder-7b-magicoder",
    train_dataset="hoangnh39/magicoder_samples_qwen2.5-coder",
    lora_config=lora_parameters,
    steps_per_session=300,
    max_total_steps=1000,
    batch_size=1,
    grad_accum_steps=16,
    save_steps=20
)

print(f"Current progress: {current_progress}/1000 steps")

Token has not been saved to git credential helper.


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: zmeb030902 (hoangnh39) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Training for 300 steps (total progress: 0/1000)


config.json:   0%|          | 0.00/668 [00:00<?, ?B/s]

...

Generating train split: 0 examples [00:00, ? examples/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/1970 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
20,0.830773
40,0.605445
60,0.599160
80,0.579041
100,0.591706
120,0.587433
140,0.570501
160,0.565949
180,0.560992
200,0.551085
